# Groq Stock Price Agent — Complete Notebook

A complete, end-to-end notebook for a function-calling agent that retrieves stock prices with `yfinance` and uses Groq's OpenAI-compatible API.

## Before you run it
1. Open this notebook in VS Code.
2. Select the **`.venv313` Python 3.13 (64-bit)** kernel for this project.
3. Put a **new/valid** Groq API key in a `.env` file in the project folder. If an API key was exposed earlier, revoke it and create a replacement.
4. Run every cell from top to bottom.

Your `.env` file should contain:
```dotenv
GROQ_API_KEY=your_new_groq_api_key
LLM_MODEL=openai/gpt-oss-120b
```
Do not commit `.env` to Git.

## 1. Install dependencies

Run this cell using the notebook's selected kernel. `%pip` installs packages into that kernel's Python environment.

In [2]:
%pip install -q --upgrade openai yfinance pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Confirm the Python environment

In [4]:
import sys
print('Python executable:', sys.executable)
print('Python version:', sys.version)

if '3.13' not in sys.version:
    print('WARNING: This project was set up for Python 3.13. In VS Code, select the .venv313 kernel if available.')

Python executable: c:\MINE\A PROJECTS FOR DATA SCIENCE\sanket-agentic\.venv313\Scripts\python.exe
Python version: 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]


## 3. Load configuration and create the Groq client

The API key is read from `GROQ_API_KEY` in your environment or `.env` file. Never paste the key directly into notebook code.

In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=False)

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
MODEL = os.getenv('LLM_MODEL', 'openai/gpt-oss-120b')

if not GROQ_API_KEY:
    raise RuntimeError(
        'GROQ_API_KEY is missing. Add it to a .env file in the project folder, '
        'or set it as an environment variable, then rerun this cell.'
    )

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url='https://api.groq.com/openai/v1',
)

print('Groq client configured.')
print('Model:', MODEL)

Groq client configured.
Model: openai/gpt-oss-120b


## 4. Define validated input and output models

In [6]:
from pydantic import BaseModel, Field

class GetStockPriceArgs(BaseModel):
    ticker_symbol: str = Field(
        description='Stock ticker symbol, for example AAPL, MSFT, or RELIANCE.NS.'
    )

class StockPriceResult(BaseModel):
    ticker: str
    price: float
    currency: str
    price_type: str = 'latest available market price'

class StockPriceError(BaseModel):
    error: str

## 5. Implement the stock-price tool

This function tries `fast_info` first and then falls back to recent historical data. Historical data may represent the most recent close rather than a live quote. Yahoo Finance can temporarily return empty data because of connectivity, throttling, or service limitations.

In [7]:
import yfinance as yf

def get_stock_price(ticker_symbol: str) -> str:
    """Return the latest available stock price as a JSON string."""
    ticker_symbol = ticker_symbol.strip().upper()

    if not ticker_symbol:
        return StockPriceError(error='Ticker symbol cannot be empty.').model_dump_json()

    try:
        stock = yf.Ticker(ticker_symbol)
        price = None
        currency = None

        # First attempt: fast_info.
        try:
            fast_info = stock.fast_info
            price = fast_info.get('last_price')
            currency = fast_info.get('currency')
        except Exception:
            pass

        # Fallback: use the most recent non-null closing price.
        if price is None:
            history = stock.history(period='5d', auto_adjust=False)
            if history.empty or 'Close' not in history.columns:
                raise ValueError(
                    'Yahoo Finance returned no price data. Check your internet connection, '
                    'ticker symbol, or try again later if the service is throttling requests.'
                )
            closing_prices = history['Close'].dropna()
            if closing_prices.empty:
                raise ValueError('No valid closing prices were returned.')
            price = float(closing_prices.iloc[-1])

        # Try to obtain currency metadata if fast_info did not provide it.
        if not currency:
            try:
                currency = stock.history_metadata.get('currency')
            except Exception:
                currency = None

        if not currency:
            # Common fallback for known exchanges; metadata is preferred when available.
            if ticker_symbol.endswith(('.NS', '.BO')):
                currency = 'INR'
            else:
                currency = 'USD'

        result = StockPriceResult(
            ticker=ticker_symbol,
            price=float(price),
            currency=str(currency),
        )
        return result.model_dump_json()

    except Exception as exc:
        return StockPriceError(
            error=f'Could not fetch {ticker_symbol}: {type(exc).__name__}: {exc}'
        ).model_dump_json()

## 6. Test the stock-price tool directly

Run this before testing the LLM agent. If it returns an error, fix the data/network issue first.

In [8]:
print(get_stock_price('TSLA'))
# Indian examples: RELIANCE.NS, TCS.NS, INFY.NS

{"ticker":"TSLA","price":372.2200012207031,"currency":"USD","price_type":"latest available market price"}


## 7. Describe the tool to the model

In [9]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_stock_price',
            'description': 'Get the latest available stock price for a ticker symbol. Use this tool when the user asks for a stock price.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'ticker_symbol': {
                        'type': 'string',
                        'description': 'Ticker symbol, such as AAPL, MSFT, or RELIANCE.NS.'
                    }
                },
                'required': ['ticker_symbol'],
                'additionalProperties': False
            }
        }
    }
]

SYSTEM_PROMPT = '''You are a helpful stock-price assistant.
When the user asks for a stock price, call the get_stock_price tool.
Do not invent prices. Explain that the result is the latest available data and may be a recent closing price, not a real-time quote.
If the tool returns an error, explain the error clearly and do not fabricate a result.
Keep the answer concise and mention the ticker, price, and currency when available.'''

print('Tool schema ready:', tools[0]['function']['name'])

Tool schema ready: get_stock_price


## 8. Build the function-calling agent loop

The loop sends the conversation to Groq, executes any requested tool call locally, adds the tool result to the conversation, and asks the model to produce the final response.

In [10]:
import json

def run_agent(user_message: str, max_tool_rounds: int = 5) -> str:
    """Run one user request through the Groq tool-calling loop."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_message},
    ]

    for _ in range(max_tool_rounds):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice='auto',
            temperature=0.2,
        )

        message = response.choices[0].message
        tool_calls = message.tool_calls or []

        if not tool_calls:
            return message.content or '(The model returned an empty response.)'

        # Preserve the assistant tool-call message in the conversation.
        messages.append(message.model_dump(exclude_none=True))

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            raw_arguments = tool_call.function.arguments or '{}'

            try:
                arguments = json.loads(raw_arguments)
            except json.JSONDecodeError as exc:
                tool_result = StockPriceError(
                    error=f'The model supplied invalid tool arguments: {exc}'
                ).model_dump_json()
            else:
                if function_name == 'get_stock_price':
                    try:
                        validated = GetStockPriceArgs.model_validate(arguments)
                        tool_result = get_stock_price(validated.ticker_symbol)
                    except Exception as exc:
                        tool_result = StockPriceError(
                            error=f'Tool input or execution error: {type(exc).__name__}: {exc}'
                        ).model_dump_json()
                else:
                    tool_result = StockPriceError(
                        error=f'Unknown tool requested: {function_name}'
                    ).model_dump_json()

            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'content': tool_result,
            })

    return 'Stopped because the agent reached the maximum number of tool rounds.'

## 9. Test the complete agent

In [11]:
print(run_agent('What is the latest available stock price for AAPL?'))

The latest available data shows **AAPL** at **$347.73 USD**. This reflects the most recent closing/market price and may not be a real‑time quote.


## 10. Interactive chat loop

Run this cell to chat with the agent. Type `exit`, `quit`, or `q` to stop.

In [12]:
def main():
    print('Stock Price Agent is ready. Type exit to stop.')
    while True:
        try:
            user_input = input('\nYou: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nGoodbye!')
            break

        if user_input.lower() in {'exit', 'quit', 'q'}:
            print('Goodbye!')
            break
        if not user_input:
            continue

        try:
            print('\nAgent:', run_agent(user_input))
        except Exception as exc:
            print(f'\nAgent error: {type(exc).__name__}: {exc}')
            print('Check your API key, model name, internet connection, and Groq service status.')

# Uncomment the next line to start interactive mode in this notebook.
# main()

## Troubleshooting

- **`GROQ_API_KEY is missing`**: check that `.env` is in the current project folder and contains `GROQ_API_KEY=...`; rerun the configuration cell.
- **`No price data` / empty history**: check connectivity and the ticker symbol; Yahoo Finance may throttle requests. Wait before retrying.
- **`NameError` for a function or class**: restart the kernel and run all cells from the top in order.
- **Wrong Python version**: use VS Code's kernel picker to select the project's `.venv313` environment.
- **Groq model/tool error**: verify the model name in `.env` and check that the model supports tool calling on your account.
- **Git safety**: add `.env` to `.gitignore`; never commit API keys.

This notebook reports the latest price data Yahoo Finance provides; it does not guarantee exchange-grade real-time quotes.